# Session 1 — CPU: Preprocess + Balanced Random Forest

**Runtime: CPU** (no GPU needed)

Steps:
1. Mount Google Drive
2. Clone / pull latest code
3. Install dependencies
4. Load secrets
5. Preprocess ERA5 data → save cache to Drive
6. Train Balanced Random Forest

After this notebook finishes → open `02_GPU_Train.ipynb` on an **H100 GPU** runtime.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## 2. Clone / pull latest code

In [ ]:
import os

REPO_URL    = 'https://github.com/MCTEEKUNG/Heatwave_Backend_Elysia.git'
REPO_DIR    = '/content/Heatwave-AI'
BRANCH      = 'feat/eda-wbgt-year-split'
TRAIN_DIR   = f'{REPO_DIR}/Heatwave-AI-TRAIN'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(TRAIN_DIR)
print(f'Working directory: {os.getcwd()}')
!git log --oneline -3

## 3. Install dependencies

In [ ]:
!pip install -r requirements.txt -q
print('Dependencies installed.')

## 4. Load secrets

Make sure these are set in **Colab Secrets** (left sidebar → key icon):
- `HF_TOKEN`
- `RENDER_API_KEY`
- `RENDER_SERVICE_ID`

In [ ]:
from google.colab import userdata
import os

os.environ['HF_TOKEN']           = userdata.get('HF_TOKEN')
os.environ['RENDER_API_KEY']     = userdata.get('RENDER_API_KEY')
os.environ['RENDER_SERVICE_ID']  = userdata.get('RENDER_SERVICE_ID')

# Authenticate HuggingFace CLI
!hf auth login --token $HF_TOKEN
print('Secrets loaded.')

## 5. Check ERA5 data path

ตรวจสอบว่า ERA5 data อยู่ใน Google Drive และ path ถูกต้อง

In [ ]:
import os

# ERA5 data should be on Google Drive
ERA5_PATH = '/content/drive/MyDrive/Heatwave-AI/Era5-data-2000-2026'

if os.path.exists(ERA5_PATH):
    files = os.listdir(ERA5_PATH)
    print(f'ERA5 path OK: {len(files)} files found')
    print('Sample:', files[:5])
else:
    print(f'ERA5 path NOT FOUND: {ERA5_PATH}')
    print('Upload your ERA5 data to Google Drive first.')

# NDVI data
NDVI_PATH = '/content/drive/MyDrive/Heatwave-AI/ndvi'
if os.path.exists(NDVI_PATH):
    print(f'NDVI path OK')
else:
    print(f'NDVI path NOT FOUND (optional if NDVI disabled in config)')

## 6. Preprocess — load ERA5, feature engineering, split, save cache

ใช้เวลาประมาณ **15–30 นาที** ขึ้นกับขนาดข้อมูล

Cache จะถูกบันทึกไปที่ `/content/drive/MyDrive/Heatwave-AI/cache/`

In [ ]:
!python train_split.py --phase preprocess --config config/config.yaml

## 7. Train Balanced Random Forest

ใช้เวลาประมาณ **10–20 นาที** บน CPU

BRF ใช้ CPU เพราะ `imbalanced-learn` ยังไม่รองรับ GPU

In [ ]:
!python train_split.py --phase brf --config config/config.yaml

## เสร็จแล้ว!

**ขั้นตอนต่อไป:**
1. เปิด notebook ใหม่: `02_GPU_Train.ipynb`
2. เปลี่ยน Runtime เป็น **T4 GPU** หรือ **H100 GPU**
   - `Runtime` → `Change runtime type` → `GPU` → `H100` (ถ้ามี)
3. รัน notebook นั้น

Cache ที่บันทึกไว้ใน Drive จะถูก reuse โดยอัตโนมัติ ไม่ต้อง preprocess ใหม่